# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [ ]:
from pathlib import Path
import os
# Load .env from notebook folder (if present) without external deps
env_path = Path('.') / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' in line:
            k,v = line.split('=',1)
            os.environ.setdefault(k.strip(), v.strip().strip("").strip("'"))
# Confirm key is available
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or the environment'

import asyncio
import json
import time
from pathlib import Path

import pandas as pd
import httpx

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

# We'll call the OpenAI chat Completions HTTP endpoint via httpx for an
# explicit asyncio + httpx pattern.
MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

async def httpx_chat(messages, model, temperature=0.0, timeout=60.0, max_retries: int = 6):
    import random
    api_key = os.environ.get('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY not set')
    base = os.environ.get('OPENAI_BASE_URL') or 'https://api.openai.com/v1'
    url = f"{base.rstrip('/')}/chat/completions"
    payload = {'model': model, 'messages': messages, 'temperature': temperature}
    headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
    attempt = 0
    while True:
        attempt += 1
        async with httpx.AsyncClient(timeout=timeout) as client:
            resp = await client.post(url, json=payload, headers=headers)
            if resp.status_code == 429 and attempt < max_retries:
                backoff = (2 ** (attempt - 1)) + random.random()
                await asyncio.sleep(backoff)
                continue
            try:
                resp.raise_for_status()
            except httpx.HTTPStatusError:
                text = resp.text if resp is not None else ''
                raise RuntimeError(f'HTTP {resp.status_code}: {text}')
            return resp.json()

# store last judge raw texts for debugging
JUDGE_DEBUG = {}

print('Setup complete.')

## Step 1 — Load the data

In [ ]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [ ]:
def _build_messages(instruction: str) -> list:
    return [
        {'role': 'system', 'content': 'You are a helpful assistant. Always respond with valid JSON when asked.'},
        {'role': 'user', 'content': instruction},
    ]

STRATEGIES = {
    'explicit_json': lambda snippet: _build_messages(
        "Extract the following fields from the job description and return only a JSON object with keys: title, company, location.\n\nJob description:\n" + snippet
    ),
    'concise': lambda snippet: _build_messages(
        "Provide a JSON object with keys 'title','company','location' extracted from the text below. Keep it minimal.\n\n" + snippet
    ),
    'few_shot': lambda snippet: _build_messages(
        "Example: {\"title\": \"Software Engineer\", \"company\": \"Acme\", \"location\": \"Remote\"}\nNow extract the same keys from the following job description and return only the JSON:\n\n" + snippet
    ),
    'structured_prompt': lambda snippet: _build_messages(
        "Extract title, company and location and return JSON. If a field is missing, use null.\n\nText:\n" + snippet
    ),
}

print('Defined STRATEGIES:', list(STRATEGIES.keys()))

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [ ]:
def parse_response(text: str) -> dict | None:
    import re, json
    if not text:
        return None
    s = text.strip()
    s = re.sub(r'```(?:json)?\n?', '', s, flags=re.IGNORECASE).replace('```', '').strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    first_obj = s.find('{')
    last_obj = s.rfind('}')
    if first_obj != -1 and last_obj != -1 and last_obj > first_obj:
        cand = s[first_obj:last_obj+1]
        try:
            return json.loads(cand)
        except Exception:
            pass
    first_arr = s.find('[')
    last_arr = s.rfind(']')
    if first_arr != -1 and last_arr != -1 and last_arr > first_arr:
        cand = s[first_arr:last_arr+1]
        try:
            return json.loads(cand)
        except Exception:
            pass
    m = re.search(r'\{.*?\}', s, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    import time
    snippet_text = snippet.get('text') or snippet.get('snippet') or snippet.get('content') or snippet.get('description') or snippet.get('job_description') or str(snippet)
    messages = STRATEGIES[strategy_name](snippet_text)
    start = time.perf_counter()
    try:
        resp = await httpx_chat(messages, MODEL, temperature=TEMPERATURE)
    except Exception as e:
        end = time.perf_counter()
        return {
            'strategy': strategy_name,
            'snippet_id': snippet.get('id'),
            'error': str(e),
            'raw': None,
            'extracted': None,
            'cost_usd': None,
            'latency_s': end - start,
        }
    end = time.perf_counter()
    latency = end - start
    raw = None
    try:
        raw = resp['choices'][0]['message']['content']
    except Exception:
        try:
            raw = resp['choices'][0]['message'].get('content') if isinstance(resp, dict) else None
        except Exception:
            raw = None
    cost = 0.0
    try:
        usage = resp.get('usage') if isinstance(resp, dict) else None
        if usage:
            prompt_t = usage.get('prompt_tokens') or usage.get('input_tokens') or 0
            comp_t = usage.get('completion_tokens') or usage.get('output_tokens') or 0
            rates = RATES.get(MODEL, {})
            in_rate = rates.get('in', 0)
            out_rate = rates.get('out', 0)
            cost = prompt_t * in_rate + comp_t * out_rate
    except Exception:
        cost = None
    extracted = parse_response(raw) if raw else None
    return {
        'strategy': strategy_name,
        'snippet_id': snippet.get('id'),
        'raw': raw,
        'extracted': extracted,
        'cost_usd': cost,
        'latency_s': latency,
    }


async def run_all(max_concurrency: int = 8) -> list[dict]:
    sem = asyncio.Semaphore(max_concurrency)
    async def run_with_sem(strat, snippet):
        async with sem:
            try:
                return await run_one(strat, snippet)
            except Exception as e:
                return {
                    'strategy': strat,
                    'snippet_id': snippet.get('id') if isinstance(snippet, dict) else None,
                    'error': f'run_one exception: {e}',
                    'raw': None,
                    'extracted': None,
                    'cost_usd': None,
                    'latency_s': None,
                }
    tasks = []
    strategies = globals().get('STRATEGIES')
    if strategies is None:
        raise RuntimeError('STRATEGIES is not defined — run the cell that defines prompt strategies before run_all')
    for snippet in snippets:
        for strat in list(strategies.keys()):
            tasks.append(asyncio.create_task(run_with_sem(strat, snippet)))
    raw_results = await asyncio.gather(*tasks, return_exceptions=True)
    results = []
    for item in raw_results:
        if isinstance(item, Exception):
            results.append({'strategy': 'unknown', 'snippet_id': None, 'error': f'Exception: {item}', 'raw': None, 'extracted': None, 'cost_usd': None, 'latency_s': None})
        else:
            results.append(item)
    return results

In [ ]:
# Run it (reduced concurrency to avoid 429 rate limits)
results = await run_all(max_concurrency=2)
print(f'Got {len(results)} results.')

# Diagnostic summary: show error counts, parsed counts, and a few samples
errs = [r for r in results if r.get('error')]
parsed = [r for r in results if r.get('extracted') is not None]
print(f'Errors: {len(errs)}; Parsed: {len(parsed)}')
if errs:
    print('Some errors (first 5):')
    for e in errs[:5]:
        print('-', e.get('strategy'), e.get('snippet_id'), '|', e.get('error'))
if results:
    print('\nSample results (first 3):')
    for r in results[:3]:
        print(r.get('strategy'), r.get('snippet_id'), 'parsed=', r.get('extracted') is not None, 'cost=', r.get('cost_usd'), 'latency=', round(r.get('latency_s') or 0, 3))
results[0]

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
def score_accuracy(extracted: dict | None, gold: dict | None) -> int:
    if not extracted or not gold:
        return 0
    def norm(v):
        return str(v).strip().lower() if v is not None else ''
    keys = ['title','company','location']
    score = 0
    for k in keys:
        e = extracted.get(k) if isinstance(extracted, dict) else None
        g = gold.get(k) if isinstance(gold, dict) else None
        if e is None or g is None:
            continue
        if norm(e) == norm(g):
            score += 1
    return score

print('Defined score_accuracy')

In [ ]:
import httpx
scored = []

def score_llm_judge_sync(snippet_text: str, extracted: dict | None, gold: dict) -> tuple[int, str | None]:
    if not extracted:
        return 1, None
    api_key = os.environ.get('OPENAI_API_KEY')
    base = os.environ.get('OPENAI_BASE_URL') or 'https://api.openai.com/v1'
    url = f"{base.rstrip('/')}/chat/completions"
    headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
    system_msg = 'You are an objective evaluator. Return ONLY valid JSON with keys "score" (int 1-25) and "explanation" (short string).'
    user_msg = (
        'Gold: ' + str(gold)
        + '\nExtracted: ' + str(extracted)
        + '\nSnippet: ' + (snippet_text or '')
        + '\n\nRubric: 25 = all fields exactly correct; lower scores reflect partial matches, minor errors, or fabrication. Return ONLY the JSON with integer "score" between 1 and 25 and a short "explanation".'
    )
    payload = {'model': JUDGE_MODEL, 'messages': [{'role': 'system', 'content': system_msg}, {'role': 'user', 'content': user_msg}], 'temperature': 0.0}
    try:
        resp = httpx.post(url, json=payload, headers=headers, timeout=60.0)
        resp.raise_for_status()
        jr = resp.json()
        try:
            text = jr['choices'][0]['message']['content']
        except Exception:
            try:
                text = jr['choices'][0]['message'].get('content') if isinstance(jr, dict) else None
            except Exception:
                text = None
        try:
            key = gold.get('id') if isinstance(gold, dict) else None
            if not key:
                key = snippet_text or None
            if key:
                JUDGE_DEBUG[key] = text if text is not None else repr(jr)[:1000]
            if globals().get('JUDGE_DEBUG_COUNT', 0) < 5:
                print('DEBUG judge raw for key=', key, '->', repr(jr)[:1000])
                globals()['JUDGE_DEBUG_COUNT'] = globals().get('JUDGE_DEBUG_COUNT', 0) + 1
        except Exception:
            pass
        parsed = parse_response(text or '')
        if isinstance(parsed, dict) and 'score' in parsed:
            try:
                s = int(parsed['score'])
                return max(1, min(25, s)), (text or (repr(jr) if jr is not None else None))
            except Exception:
                pass
        import re
        nums = re.findall(r"\d+", (text or (repr(jr) if jr is not None else '')))
        for n in nums:
            try:
                v = int(n)
                if 1 <= v <= 25:
                    return v, (text or (repr(jr) if jr is not None else None))
            except Exception:
                continue
    except Exception as e:
        try:
            key = gold.get('id') if isinstance(gold, dict) else None
            if not key:
                key = snippet_text or None
            if key:
                JUDGE_DEBUG[key] = f'ERROR: {e}'
        except Exception:
            pass
        return 1, (JUDGE_DEBUG.get(key) if key else repr(e))
    return 1, (text or (repr(jr) if jr is not None else None))

for r in results:
    snippet_id = r.get('snippet_id')
    gold_entry = golden.get(snippet_id) if 'golden' in globals() else None
    acc = score_accuracy(r.get('extracted'), gold_entry) if gold_entry else 0
    parse_ok = (r.get('extracted') is not None)
    judge = 1
    raw = None
    if gold_entry:
        try:
            s, raw = score_llm_judge_sync(gold_entry.get('snippet') or '', r.get('extracted'), gold_entry)
            judge = int(s)
        except Exception as e:
            try:
                if snippet_id:
                    JUDGE_DEBUG[snippet_id] = f'ERROR: {e}'
            except Exception:
                pass
            judge = 1
    row = dict(r)
    row['accuracy'] = acc
    row['parse_success'] = parse_ok
    row['llm_judge_score'] = judge
    try:
        row['llm_judge_raw'] = raw or JUDGE_DEBUG.get(snippet_id) or (gold_entry.get('id') and JUDGE_DEBUG.get(gold_entry.get('id')))
    except Exception:
        row['llm_judge_raw'] = None
    scored.append(row)

print(f'Scored {len(scored)} results.')
raws = [r.get('llm_judge_raw') for r in scored if r.get('llm_judge_raw')]
print('Judge raw count:', len(raws))
for i, v in enumerate(raws[:10]):
    print('--- raw', i, v[:400] if isinstance(v, str) else v)

print('\nSample scored rows (first 6):')
for r in scored[:6]:
    print(r.get('strategy'), r.get('snippet_id'), 'judge_raw=', (r.get('llm_judge_raw') or '')[:200], '-> judge_score=', r.get('llm_judge_score'))

from collections import Counter
parsed_scores = []
import re
for r in scored:
    txt = r.get('llm_judge_raw')
    if not txt:
        parsed_scores.append(None)
        continue
    p = parse_response(txt)
    if isinstance(p, dict) and 'score' in p:
        try:
            parsed_scores.append(int(p['score']))
            continue
        except Exception:
            pass
    m = re.search(r"\d+", txt)
    if m:
        parsed_scores.append(int(m.group()))
    else:
        parsed_scores.append(None)

print('Parsed judge score counts:', Counter(parsed_scores))

## Step 5 — Build the comparison table

In [ ]:
df = pd.DataFrame(scored)
if df.empty:
    print('No scored rows to summarize. Ensure scoring ran successfully.')
    summary = pd.DataFrame()
else:
    if 'strategy' not in df.columns:
        df['strategy'] = 'unknown'
    for col, default in [('accuracy', 0), ('parse_success', False), ('llm_judge_score', 1), ('cost_usd', 0.0), ('latency_s', 0.0)]:
        if col not in df.columns:
            df[col] = default
    df['parse_success'] = df['parse_success'].fillna(False).astype(bool)
    df['llm_judge_score'] = pd.to_numeric(df['llm_judge_score'], errors='coerce').fillna(1).astype(float)
    df['accuracy'] = pd.to_numeric(df['accuracy'], errors='coerce').fillna(0).astype(float)
    df['cost_usd'] = pd.to_numeric(df['cost_usd'], errors='coerce').fillna(0.0).astype(float)
    df['latency_s'] = pd.to_numeric(df['latency_s'], errors='coerce').fillna(0.0).astype(float)
    try:
        summary = df.groupby('strategy').agg({
            'accuracy': 'mean',
            'parse_success': 'mean',
            'llm_judge_score': 'mean',
            'cost_usd': 'sum',
            'latency_s': 'median',
        }).round(6)
        summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
    except KeyError as e:
        print('Grouping failed — dataframe columns:', list(df.columns), 'error:', e)
        summary = pd.DataFrame()

summary

## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```